# Barcelona Noise Prediction: Building Use Feature Engineering

This notebook evaluates the `currentUse` attribute of Cadastral buildings to derive usage percentages around our street network.
We load the `bcn_catastral_building.gpkg` layer and calculate the proportion of each building use (e.g., residential, retail, industrial) within 50m and 100m street buffers.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import os

print("Loading spatial data...")
# Load streets
noise_streets = gpd.read_file("../../layers/BCN_noise_streets.gpkg")
noise_streets['fid'] = noise_streets.index

# Load cadastral buildings
buildings = gpd.read_file("../../layers/bcn_catastral_building.gpkg")

# Ensure CRS alignment
buildings = buildings.to_crs(noise_streets.crs)

print(f"Loaded {len(noise_streets)} streets and {len(buildings)} buildings.")
display(buildings[['gml_id', 'currentUse', 'geometry']].head())

## Clean and Map Building Uses

The Cadastral `currentUse` has predefined codes like `1_residential` and `4_2_retail`. We can simplify these into machine-learning friendly categories.

In [ ]:
print(buildings['currentUse'].value_counts(dropna=False))

# Mapping dictionary for cleaner feature names
use_mapping = {
    '1_residential': 'residential',
    '4_2_retail': 'retail',
    '4_3_publicServices': 'public_services',
    '3_industrial': 'industrial',
    '4_1_office': 'office',
    '2_agriculture': 'agriculture'
}

# Apply mapping, filling NaNs or unmapped with 'unknown'
buildings['ml_use'] = buildings['currentUse'].map(use_mapping).fillna('unknown')

fig, ax = plt.subplots(figsize=(8, 5))
buildings['ml_use'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black', ax=ax)
ax.set_title("Distribution of Building Uses")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.show()

## Calculate Buffer Area Percentages

We use spatial overlays to calculate what percentage of the buffer zone is occupied by each building use type.

In [ ]:
def calculate_building_use_pct(streets_gdf, bldgs_gdf, buffer_size):
    print(f"Processing {buffer_size}m buffers...")
    
    # Create buffers and compute their total area
    buffered = streets_gdf[['fid', 'TRAM', 'geometry']].copy()
    buffered['geometry'] = buffered.geometry.buffer(buffer_size)
    buffered['buffer_area'] = buffered.geometry.area
    
    # Optional performance optimization: keep only needed columns in buildings
    b_reduced = bldgs_gdf[['ml_use', 'geometry']].copy()
    
    # Intersect buffers with building footprints to find exact overlapping areas
    intersection = gpd.overlay(buffered, b_reduced, how='intersection')
    intersection['intersected_area'] = intersection.geometry.area
    
    # Group by street identifier and building use, then sum up intersected area
    grouped = intersection.groupby(['fid', 'TRAM', 'ml_use'])['intersected_area'].sum().reset_index()
    
    # Merge buffer total area back in to divide and derive percentages
    grouped = grouped.merge(buffered[['fid', 'TRAM', 'buffer_area']], on=['fid', 'TRAM'])
    grouped['pct'] = (grouped['intersected_area'] / grouped['buffer_area']) * 100
    
    # Pivot into wide format for machine learning integration
    features = grouped.pivot(index=['fid', 'TRAM'], columns='ml_use', values='pct').fillna(0)
    
    # Tag column names with the buffer size designation
    features.columns = [f"bldg_{col}_pct_{buffer_size}m" for col in features.columns]
    
    return features.reset_index()

features_50m = calculate_building_use_pct(noise_streets, buildings, 50)
features_100m = calculate_building_use_pct(noise_streets, buildings, 100)

display(features_50m.head())

## Compile Final Dataset and Export

Merge the percentages generated back onto our core layout indexed by `fid` and `TRAM`, and export the DataFrame.

In [ ]:
# Baseline dataset structure
dataset = pd.DataFrame({
    'fid': noise_streets['fid'],
    'street_id': noise_streets['TRAM']
})

# Attach all features
dataset = dataset.merge(features_50m, left_on=['fid', 'street_id'], right_on=['fid', 'TRAM'], how='left').drop(columns=['TRAM'], errors='ignore')
dataset = dataset.merge(features_100m, left_on=['fid', 'street_id'], right_on=['fid', 'TRAM'], how='left').drop(columns=['TRAM'], errors='ignore')

dataset = dataset.fillna(0)
display(dataset.head())

# Export the engineered feature CSV
output_dir = "../../data/processed"
os.makedirs(output_dir, exist_ok=True)
dataset.to_csv(os.path.join(output_dir, "building_use_features.csv"), index=False)
print("Successfully generated and exported: building_use_features.csv")

## Mapping Output Results

Let's visualize the resulting physical footprint to verify our generated residential concentrations along the streets.

In [ ]:
viz_gdf = noise_streets.merge(dataset[['fid', 'bldg_residential_pct_50m']], on='fid')

fig, ax = plt.subplots(figsize=(12, 10))

# Cap at the 95th percentile so localized huge residential blocks don't fade out smaller blocks visually
vmax_val = viz_gdf['bldg_residential_pct_50m'].quantile(0.95)

viz_gdf.plot(
    column='bldg_residential_pct_50m',
    ax=ax,
    cmap='OrRd',
    linewidth=1.2,
    legend=True,
    legend_kwds={'label': "Residential Building Coverage % (50m Buffer)"},
    vmax=vmax_val
)

ax.set_title("Barcelona Street Network: Residential Building Concentration", fontsize=16)
ax.set_axis_off()
plt.show()